# OMOP Drug Exposure Table

Transforms FHIR MedicationRequest/MedicationAdministration resources into OMOP CDM `drug_exposure` table.

## Mapping: FHIR MedicationRequest → OMOP Drug_Exposure

| OMOP Field | FHIR Source | Transformation |
|------------|-------------|----------------|
| drug_exposure_id | MedicationRequest.id | Hash to integer |
| person_id | MedicationRequest.subject | Reference to person |
| drug_concept_id | MedicationRequest.medicationCodeableConcept | Map RxNorm to OMOP |
| drug_exposure_start_date | MedicationRequest.authoredOn | Extract date |
| drug_exposure_start_datetime | MedicationRequest.authoredOn | Full timestamp |
| drug_exposure_end_date | Calculated | Start + dispense duration |
| drug_type_concept_id | - | 38000177 (Prescription written) |
| quantity | MedicationRequest.dispenseRequest.quantity | Dispensed amount |
| days_supply | MedicationRequest.dispenseRequest.expectedSupplyDuration | Supply days |
| sig | MedicationRequest.dosageInstruction | Dosage text |
| route_concept_id | MedicationRequest.dosageInstruction.route | Route concept |
| drug_source_value | MedicationRequest.medicationCodeableConcept.code | Original code |

## Important Notes

- FHIR MedicationRequest represents a **prescription order**, not actual medication exposure
- Use `drug_type_concept_id = 38000177` (Prescription written) to indicate this distinction
- RxNorm is the standard vocabulary for medications in OMOP

_Note: Attach to a Serverless SQL Warehouse for execution._

In [ ]:
-- ============================================================================
-- CONFIGURATION
-- ============================================================================
DECLARE OR REPLACE VARIABLE catalog_use STRING DEFAULT 'redox_fhir';
DECLARE OR REPLACE VARIABLE silver_schema STRING DEFAULT 'bronze';
DECLARE OR REPLACE VARIABLE gold_schema STRING DEFAULT 'omop';

SET VARIABLE catalog_use = COALESCE(:catalog_use, catalog_use);
SET VARIABLE silver_schema = COALESCE(:silver_schema, silver_schema);
SET VARIABLE gold_schema = COALESCE(:gold_schema, gold_schema);

USE IDENTIFIER(catalog_use || '.' || gold_schema);
SELECT current_catalog(), current_schema();

## Create Drug Exposure Streaming Table

In [ ]:
DECLARE OR REPLACE VARIABLE create_drug_stmt STRING;

SET VARIABLE create_drug_stmt = "
CREATE OR REFRESH STREAMING TABLE drug_exposure (
  -- Primary key
  drug_exposure_id BIGINT NOT NULL COMMENT 'Unique drug exposure identifier'
  
  -- Person reference
  ,person_id BIGINT NOT NULL COMMENT 'Reference to person table'
  
  -- Drug coding
  ,drug_concept_id INT NOT NULL COMMENT 'OMOP standard concept for drug (RxNorm)'
  
  -- Dates
  ,drug_exposure_start_date DATE NOT NULL COMMENT 'Drug exposure start date'
  ,drug_exposure_start_datetime TIMESTAMP COMMENT 'Drug exposure start datetime'
  ,drug_exposure_end_date DATE COMMENT 'Drug exposure end date'
  ,drug_exposure_end_datetime TIMESTAMP COMMENT 'Drug exposure end datetime'
  ,verbatim_end_date DATE COMMENT 'Stated end date'
  
  -- Type
  ,drug_type_concept_id INT NOT NULL DEFAULT 38000177 COMMENT 'Type: 38000177=Prescription written'
  
  -- Prescription details
  ,stop_reason STRING COMMENT 'Reason drug was stopped'
  ,refills INT COMMENT 'Number of refills'
  ,quantity DECIMAL(10,2) COMMENT 'Quantity dispensed'
  ,days_supply INT COMMENT 'Days of supply'
  ,sig STRING COMMENT 'Dosage instructions (sig)'
  
  -- Route
  ,route_concept_id INT DEFAULT 0 COMMENT 'Route of administration concept'
  ,route_source_value STRING COMMENT 'Original route value'
  
  -- Lot
  ,lot_number STRING COMMENT 'Manufacturer lot number'
  
  -- References
  ,provider_id BIGINT COMMENT 'Reference to provider table'
  ,visit_occurrence_id BIGINT COMMENT 'Reference to visit_occurrence table'
  ,visit_detail_id BIGINT COMMENT 'Reference to visit_detail table'
  
  -- Source values
  ,drug_source_value STRING COMMENT 'Original drug code'
  ,drug_source_concept_id INT DEFAULT 0 COMMENT 'Source vocabulary concept'
  
  -- Drug info
  ,drug_code_system STRING COMMENT 'Source code system (RxNorm, NDC, etc.)'
  ,drug_display STRING COMMENT 'Display text for drug'
  
  -- Lineage
  ,fhir_medicationrequest_uuid STRING COMMENT 'Original FHIR MedicationRequest UUID'
  ,bundle_uuid STRING COMMENT 'Source bundle reference'
)
COMMENT 'OMOP CDM Drug Exposure table - Medications from FHIR MedicationRequest resources'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true'
  ,'delta.enableDeletionVectors' = 'true'
  ,'delta.enableRowTracking' = 'true'
  ,'quality' = 'gold'
  ,'pipelines.channel' = 'PREVIEW'
  ,'delta.feature.variantType-preview' = 'supported'
)
AS 
SELECT
  -- Generate integer drug_exposure_id
  ABS(HASH(COALESCE(id::STRING, medicationrequest_uuid))) AS drug_exposure_id
  
  -- Person reference
  ,ABS(HASH(
    COALESCE(
      REGEXP_EXTRACT(subject:reference::STRING, 'Patient/(.+)', 1),
      subject:reference::STRING
    )
  )) AS person_id
  
  -- Drug concept - placeholder using hash of code
  -- In production, join to OMOP vocabulary tables for proper RxNorm concept_id
  ,COALESCE(
    ABS(HASH(medicationCodeableConcept:coding[0]:code::STRING)) % 2000000000,
    0
  ) AS drug_concept_id
  
  -- Dates
  ,COALESCE(
    CAST(TRY_CAST(authoredOn::STRING AS TIMESTAMP) AS DATE),
    CURRENT_DATE()
  ) AS drug_exposure_start_date
  ,TRY_CAST(authoredOn::STRING AS TIMESTAMP) AS drug_exposure_start_datetime
  
  -- Calculate end date from days supply if available
  ,CASE 
    WHEN dispenseRequest:expectedSupplyDuration:value IS NOT NULL THEN
      DATE_ADD(
        COALESCE(CAST(TRY_CAST(authoredOn::STRING AS TIMESTAMP) AS DATE), CURRENT_DATE()),
        CAST(dispenseRequest:expectedSupplyDuration:value::STRING AS INT)
      )
    ELSE NULL
  END AS drug_exposure_end_date
  ,NULL AS drug_exposure_end_datetime
  ,NULL AS verbatim_end_date
  
  -- Type (Prescription written for MedicationRequest)
  ,38000177 AS drug_type_concept_id
  
  -- Prescription details
  ,NULL AS stop_reason
  ,TRY_CAST(dispenseRequest:numberOfRepeatsAllowed::STRING AS INT) AS refills
  ,TRY_CAST(dispenseRequest:quantity:value::STRING AS DECIMAL(10,2)) AS quantity
  ,TRY_CAST(dispenseRequest:expectedSupplyDuration:value::STRING AS INT) AS days_supply
  ,dosageInstruction[0]:text::STRING AS sig
  
  -- Route
  ,0 AS route_concept_id
  ,dosageInstruction[0]:route:coding[0]:code::STRING AS route_source_value
  
  -- Lot
  ,NULL AS lot_number
  
  -- References
  ,CASE 
    WHEN requester:reference IS NOT NULL THEN
      ABS(HASH(REGEXP_EXTRACT(requester:reference::STRING, 'Practitioner/(.+)', 1)))
    ELSE NULL
  END AS provider_id
  ,CASE 
    WHEN encounter:reference IS NOT NULL THEN
      ABS(HASH(REGEXP_EXTRACT(encounter:reference::STRING, 'Encounter/(.+)', 1)))
    ELSE NULL
  END AS visit_occurrence_id
  ,NULL AS visit_detail_id
  
  -- Source values
  ,medicationCodeableConcept:coding[0]:code::STRING AS drug_source_value
  ,0 AS drug_source_concept_id
  
  -- Drug info
  ,medicationCodeableConcept:coding[0]:system::STRING AS drug_code_system
  ,COALESCE(
    medicationCodeableConcept:coding[0]:display::STRING,
    medicationCodeableConcept:text::STRING
  ) AS drug_display
  
  -- Lineage
  ,medicationrequest_uuid AS fhir_medicationrequest_uuid
  ,bundle_uuid
  
FROM STREAM(" || catalog_use || "." || silver_schema || ".medicationrequest)
WHERE status::STRING NOT IN ('cancelled', 'entered-in-error')
";

SELECT create_drug_stmt AS statement;

In [ ]:
EXECUTE IMMEDIATE create_drug_stmt;

In [ ]:
-- Verify drug_exposure table
SELECT 
  drug_exposure_id,
  person_id,
  drug_concept_id,
  drug_exposure_start_date,
  days_supply,
  drug_source_value,
  drug_display
FROM drug_exposure
LIMIT 10;

In [ ]:
-- Drug vocabulary distribution
SELECT 
  CASE 
    WHEN drug_code_system LIKE '%rxnorm%' THEN 'RxNorm'
    WHEN drug_code_system LIKE '%ndc%' THEN 'NDC'
    ELSE drug_code_system
  END AS vocabulary,
  COUNT(*) AS count
FROM drug_exposure
GROUP BY vocabulary
ORDER BY count DESC;

In [ ]:
-- Top prescribed medications
SELECT 
  drug_source_value,
  drug_display,
  COUNT(*) AS prescriptions
FROM drug_exposure
GROUP BY drug_source_value, drug_display
ORDER BY prescriptions DESC
LIMIT 20;